# 02 — Simple Data Cleaning Pipeline
A beginner-friendly walkthrough of all 20 cleaning rules.  
Each step follows: **SEE → CLEAN → CHECK**

---

In [297]:
# ── SETUP ──
import pandas as pd
import numpy as np
import re
import os
import hashlib
from datetime import datetime, timedelta

# Load raw data
df = pd.read_csv('../datasets/full/raw_cems_data.csv')
sensors = pd.read_csv('../datasets/full/sensor_master.csv')
maint = pd.read_csv('../datasets/full/maintenance_logs.csv')
manual = pd.read_csv('../datasets/full/manual_entries.csv')
thresholds = pd.read_csv('../datasets/full/regulatory_thresholds.csv')

# Output directory
os.makedirs('../cleaned/full', exist_ok=True)

# Audit log — every change gets recorded here
audit_log = []
original_row_count = len(df)  # save for summary

print(f'Loaded {len(df)} rows, {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')
print(f'\nFirst 3 rows:')
print(df.head(3).to_string())

Loaded 27121 rows, 11 columns
Columns: ['Record_ID', 'Plant_ID', 'Stack_ID', 'Flow_Rate_m3_hr', 'TS', 'PM2.5', 'SO2', 'NOx', 'Unit', 'Status', 'Lat_Lon']

First 3 rows:
  Record_ID Plant_ID Stack_ID  Flow_Rate_m3_hr                TS  PM2.5    SO2    NOx    Unit Status          Lat_Lon
0    E00001    PL-01     S-01           2033.5  01-01-2025 00:00  170.3  119.3  103.4   ug/m3   down   13.083,80.2702
1    E00002    PL-01     S-02           3377.9  01-01-2025 00:00   <2.0   94.2   31.6   µg/m³  MAINT  13.0839,80.2721
2    E00003    PL-02     S-01           3798.4  01-01-2025 00:00  123.9    NaN  101.3  mg/Nm3  FAULT  19.0763,72.8777


---
## PHASE A: Text Normalization
Fix string values before anything else.

### Rule 2 — Standardize Unit Strings
**Why?** Some sensors write `µg/m³` (Unicode) instead of `ug/m3` (ASCII).  
They mean the same thing, but the inconsistency causes problems in analysis.

In [298]:
# ── SEE: What unit values do we have? ──
print('BEFORE — Unit values:')
print(df['Unit'].value_counts())
print()

BEFORE — Unit values:
Unit
ug/m3     21753
mg/Nm3     2745
µg/m³      2623
Name: count, dtype: int64



In [299]:
# ── CLEAN: Replace unicode µg/m³ with plain ug/m3 ──
unicode_mask = df['Unit'] == '\u00b5g/m\u00b3'  # µg/m³
count = unicode_mask.sum()

# log each change
for idx in df[unicode_mask].index:
    audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': 'Unit',
                      'Old': '\u00b5g/m\u00b3', 'New': 'ug/m3', 'Rule': 'R2'})

df.loc[unicode_mask, 'Unit'] = 'ug/m3'
print(f'Fixed {count} rows: \u00b5g/m\u00b3 \u2192 ug/m3')

Fixed 2623 rows: µg/m³ → ug/m3


In [300]:
# ── CHECK: All clean now? ──
print('AFTER — Unit values:')
print(df['Unit'].value_counts())
# Note: mg/Nm3 is intentional — we'll convert those values in Rule 14

AFTER — Unit values:
Unit
ug/m3     24376
mg/Nm3     2745
Name: count, dtype: int64


### Rule 3 — Trim & Normalize Status
**Why?** Status values have extra spaces (`" OK "`) and mixed case (`"ok"`, `"Ok"`).  
We strip whitespace and uppercase everything.

In [301]:
# ── SEE ──
print('BEFORE — Status values:')
print(df['Status'].value_counts())
print()

BEFORE — Status values:
Status
FAULT          6308
OK             6276
MAINT          6239
error           630
  MAINT         616
offline         606
 maint          601
 fault          595
DOWN            593
Error 404       585
  OK            577
ok              575
maintenance     574
 ok             560
OFFLINE         556
 Ok             554
down            554
Name: count, dtype: int64



In [302]:
# ── CLEAN ──
old_status = df['Status'].copy()
df['Status'] = df['Status'].astype(str).str.strip().str.upper()

# log only the ones that changed
changed = old_status.astype(str) != df['Status']
for idx in df[changed].index:
    audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': 'Status',
                      'Old': old_status[idx], 'New': df.at[idx, 'Status'], 'Rule': 'R3'})

print(f'Trimmed/normalized {changed.sum()} status values')

Trimmed/normalized 7742 status values


In [303]:
# ── CHECK ──
print('AFTER — Status values:')
print(df['Status'].value_counts())

AFTER — Status values:
Status
OK             8542
MAINT          7456
FAULT          6903
OFFLINE        1162
DOWN           1147
ERROR           630
ERROR 404       585
MAINTENANCE     574
NAN             122
Name: count, dtype: int64


---
## PHASE B: Timestamp Fixes
Timestamps must be in `DD-MM-YYYY HH:MM` before anything time-based.

### Timestamp Format Standardization
**Why?** Some timestamps use slashes (`01/03/2025`), some are ISO format (`2025-01-03 14:30:00`).  
We need one consistent format.

In [304]:
# ── SEE: Find non-standard formats ──
slash_mask = df['TS'].str.contains('/', na=False)
iso_mask = df['TS'].str.match(r'^\d{4}-', na=False)
print(f'Slash format (DD/MM/YYYY): {slash_mask.sum()} rows')
print(f'ISO format (YYYY-MM-DD):   {iso_mask.sum()} rows')

# show examples
if slash_mask.any():
    print(f'\nSlash example: {df.loc[slash_mask, "TS"].iloc[0]}')
if iso_mask.any():
    print(f'ISO example:   {df.loc[iso_mask, "TS"].iloc[0]}')

Slash format (DD/MM/YYYY): 1329 rows
ISO format (YYYY-MM-DD):   545 rows

Slash example: 01/01/2025 00:30
ISO example:   2025-01-01 01:45:00


In [305]:
# ── CLEAN: Fix slash format ──
for idx in df[slash_mask].index:
    old = df.at[idx, 'TS']
    df.at[idx, 'TS'] = old.replace('/', '-')
    audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': 'TS',
                      'Old': old, 'New': df.at[idx, 'TS'], 'Rule': 'TS_FORMAT'})

print(f'Fixed {slash_mask.sum()} slash-format timestamps')

# ── CLEAN: Fix ISO format ──
for idx in df[iso_mask].index:
    old = df.at[idx, 'TS']
    try:
        dt = datetime.strptime(old.strip(), '%Y-%m-%d %H:%M:%S')
        df.at[idx, 'TS'] = dt.strftime('%d-%m-%Y %H:%M')
        audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': 'TS',
                          'Old': old, 'New': df.at[idx, 'TS'], 'Rule': 'TS_FORMAT'})
    except:
        pass

print(f'Fixed {iso_mask.sum()} ISO-format timestamps')

Fixed 1329 slash-format timestamps
Fixed 545 ISO-format timestamps


### Rule 1 — Fix Midnight Rollover (Hour=24)
**Why?** Some sensors write midnight as `24:00` instead of `00:00` next day.  
`24:15` on Jan 5 = `00:15` on Jan 6.

In [306]:
# ── SEE ──
midnight_mask = df['TS'].str.contains(r'\b24:\d{2}', na=False, regex=True)
print(f'Midnight rollover timestamps: {midnight_mask.sum()}')
if midnight_mask.any():
    print(f'Example: {df.loc[midnight_mask, "TS"].iloc[0]}')

Midnight rollover timestamps: 797
Example: 31-12-2024 24:00


In [307]:
# ── CLEAN ──
for idx in df[midnight_mask].index:
    old = df.at[idx, 'TS']
    parts = old.strip().split(' ')
    date_str = parts[0]               # e.g. '05-01-2025'
    minute = parts[1].split(':')[1]    # e.g. '15'
    
    # add one day, set hour to 00
    dt = datetime.strptime(date_str, '%d-%m-%Y') + timedelta(days=1)
    new_ts = dt.strftime('%d-%m-%Y') + f' 00:{minute}'
    
    df.at[idx, 'TS'] = new_ts
    audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': 'TS',
                      'Old': old, 'New': new_ts, 'Rule': 'R1'})

print(f'Fixed {midnight_mask.sum()} midnight rollovers')

Fixed 797 midnight rollovers


### Rule 17 — Convert UTC to IST (+5:30)
**Why?** India Standard Time is UTC+5:30. Some sensors log in UTC.

In [308]:
# ── SEE ──
utc_mask = df['TS'].str.contains('UTC', na=False)
print(f'UTC timestamps: {utc_mask.sum()}')
if utc_mask.any():
    print(f'Example: {df.loc[utc_mask, "TS"].iloc[0]}')

UTC timestamps: 1292
Example: 31-12-2024 19:00 UTC


In [309]:
# ── CLEAN ──
for idx in df[utc_mask].index:
    old = df.at[idx, 'TS']
    clean = old.replace('UTC', '').strip()
    dt = datetime.strptime(clean, '%d-%m-%Y %H:%M')
    dt = dt + timedelta(hours=5, minutes=30)  # UTC → IST
    new_ts = dt.strftime('%d-%m-%Y %H:%M')
    
    df.at[idx, 'TS'] = new_ts
    audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': 'TS',
                      'Old': old, 'New': new_ts, 'Rule': 'R17'})

print(f'Converted {utc_mask.sum()} UTC timestamps to IST')

Converted 1292 UTC timestamps to IST


In [310]:
# ── CHECK: Parse all timestamps ──
df['TS'] = pd.to_datetime(df['TS'], format='%d-%m-%Y %H:%M')
print(f'All timestamps parsed successfully!')
print(f'Date range: {df["TS"].min()} to {df["TS"].max()}')

All timestamps parsed successfully!
Date range: 2025-01-01 00:00:00 to 2025-01-30 23:45:00


---
## PHASE C: Deduplication & Gap Filling

### Rule 6 — Remove Duplicate Rows
**Why?** Some sensors send the same reading twice. We keep the first one.

In [311]:
# ── SEE ──
print(f'BEFORE — Total rows: {len(df)}')

# ── CLEAN ──
before = len(df)
df = df.drop_duplicates(subset=['Plant_ID', 'Stack_ID', 'TS'], keep='first')
dupes_removed = before - len(df)

# ── CHECK ──
print(f'AFTER  — Total rows: {len(df)}')
print(f'Duplicates removed:  {dupes_removed}')

BEFORE — Total rows: 27121
AFTER  — Total rows: 26385
Duplicates removed:  736


### Rule 9 — Fill Record_ID Gaps
**Why?** Record IDs should be sequential (E00001, E00002, ...).  
Missing IDs mean data was lost. We insert placeholder rows so the gap is visible.

**How it works:**  
- Find which IDs are missing from the sequence  
- Insert a row for each missing ID with `NaN` values and `Status = GAP_FILLED`  
- This way, anyone looking at the data CAN SEE where data is missing

In [312]:
# ── SEE: Find gaps in Record_ID sequence ──
# extract the number from 'E00001' -> 1
record_nums = df['Record_ID'].str.replace('E', '').astype(int)
all_expected = set(range(record_nums.min(), record_nums.max() + 1))
all_actual = set(record_nums)
missing_ids = sorted(all_expected - all_actual)

print(f'Record_ID range: E{record_nums.min():05d} to E{record_nums.max():05d}')
print(f'Expected count:  {len(all_expected)}')
print(f'Actual count:    {len(all_actual)}')
print(f'Missing IDs:     {len(missing_ids)}')
if missing_ids:
    print(f'First 5 missing: {[f"E{m:05d}" for m in missing_ids[:5]]}')

Record_ID range: E00001 to E28800
Expected count:  28800
Actual count:    26385
Missing IDs:     2415
First 5 missing: ['E00011', 'E00013', 'E00090', 'E00114', 'E00119']


In [313]:
# ── CLEAN: Insert placeholder rows for missing IDs ── 
gap_rows = []
for mid in missing_ids:
    gap_rows.append({
        'Record_ID': f'E{mid:05d}',
        'Plant_ID': np.nan, 'Stack_ID': np.nan,
        'Flow_Rate_m3_hr': np.nan, 'TS': pd.NaT,
        'PM2.5': np.nan, 'SO2': np.nan, 'NOx': np.nan,
        'Unit': 'ug/m3', 'Status': 'GAP_FILLED', 'Lat_Lon': np.nan
    })
    audit_log.append({'Record_ID': f'E{mid:05d}', 'Column': 'ALL',
                      'Old': 'MISSING', 'New': 'GAP_FILLED_ROW', 'Rule': 'R9'})

if gap_rows:
    df = pd.concat([df, pd.DataFrame(gap_rows)], ignore_index=True)
    df = df.sort_values('Record_ID').reset_index(drop=True)

# ── CHECK ──
print(f'Inserted {len(gap_rows)} placeholder rows')
print(f'Total rows now: {len(df)}')
print(f'Status counts after gap fill:')
print(df['Status'].value_counts())

Inserted 2415 placeholder rows
Total rows now: 28800
Status counts after gap fill:
Status
OK             8313
MAINT          7240
FAULT          6718
GAP_FILLED     2415
OFFLINE        1136
DOWN           1117
ERROR           615
ERROR 404       571
MAINTENANCE     555
NAN             120
Name: count, dtype: int64


---
## PHASE D: Pollutant Value Cleaning
Fix the actual measurement values — negatives, BDL, and unit conversions.

### Rule 5 — Discard Negative Pollutant Values
**Why?** You can't have negative concentration of pollution. It's a sensor error.  
We replace negatives with `NaN` (missing).

In [314]:
# ── SEE & CLEAN each pollutant column ──
for col in ['PM2.5', 'SO2', 'NOx']:
    numeric_vals = pd.to_numeric(df[col], errors='coerce')
    neg_mask = numeric_vals < 0
    count = neg_mask.sum()
    
    if count > 0:
        # show a few examples
        examples = df.loc[neg_mask, col].head(3).tolist()
        print(f'{col}: {count} negatives found (e.g. {examples})')
        
        # log & fix
        for idx in df[neg_mask].index:
            audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': col,
                              'Old': df.at[idx, col], 'New': 'NaN', 'Rule': 'R5'})
        df.loc[neg_mask, col] = np.nan
        print(f'  -> Replaced with NaN')
    else:
        print(f'{col}: No negatives found')

PM2.5: 1354 negatives found (e.g. ['-1.9', '-9.4', '-6.4'])
  -> Replaced with NaN
SO2: 1351 negatives found (e.g. ['-5.5', '-11.4', '-10.3'])
  -> Replaced with NaN
NOx: 1356 negatives found (e.g. ['-8.9', '-5.6', '-4.7'])
  -> Replaced with NaN


### Rule 11 — Handle BDL (Below Detection Limit)
**Why?** When pollution is too low for the sensor to measure, it reports `BDL` or `<2.0`.  
**Convention:** Replace with `LOD / 2` (limit of detection divided by 2) from `sensor_master`.

In [315]:
# ── SEE: What BDL values exist? ──
for col in ['PM2.5', 'SO2', 'NOx']:
    vals = df[col].astype(str).str.strip().str.upper()
    bdl_mask = vals.isin(['BDL']) | vals.str.startswith('<')
    if bdl_mask.any():
        examples = df.loc[bdl_mask, col].unique()[:5]
        print(f'{col}: {bdl_mask.sum()} BDL values (e.g. {list(examples)})')

PM2.5: 1352 BDL values (e.g. ['<2.0', '< 5.0', 'BDL', '< 2.0', '< 1.0'])
SO2: 1350 BDL values (e.g. ['< 4.0', 'BDL', '< 5.0', '<4.0', '<5.0'])
NOx: 1359 BDL values (e.g. ['< 5.0', 'BDL', '<2.0', '<5.0', '< 2.0'])


In [316]:
# ── CLEAN: Replace BDL with LOD/2 ──
# First, build LOD lookup from sensor_master
lod_map = {}  # (Plant_ID, Stack_ID) -> {PM2.5: lod, SO2: lod, NOx: lod}
for _, row in sensors.iterrows():
    lod_map[(row['Plant_ID'], row['Stack_ID'])] = {
        'PM2.5': row['LOD_PM25'], 'SO2': row['LOD_SO2'], 'NOx': row['LOD_NOx']
    }

print('LOD values per sensor:')
for key, lods in lod_map.items():
    print(f'  {key}: {lods}')
print()

for col in ['PM2.5', 'SO2', 'NOx']:
    vals = df[col].astype(str).str.strip().str.upper()
    bdl_mask = vals.isin(['BDL']) | vals.str.startswith('<')
    
    for idx in df[bdl_mask].index:
        key = (df.at[idx, 'Plant_ID'], df.at[idx, 'Stack_ID'])
        lod = lod_map.get(key, {}).get(col, 2.0)
        replacement = round(lod / 2, 2)
        
        audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': col,
                          'Old': df.at[idx, col], 'New': str(replacement), 'Rule': 'R11'})
        df.at[idx, col] = str(replacement)
    
    print(f'{col}: {bdl_mask.sum()} BDL values replaced with LOD/2')

LOD values per sensor:
  ('PL-01', 'S-01'): {'PM2.5': 2.0, 'SO2': 4.0, 'NOx': 5.0}
  ('PL-01', 'S-02'): {'PM2.5': 2.0, 'SO2': 4.0, 'NOx': 5.0}
  ('PL-02', 'S-01'): {'PM2.5': 2.0, 'SO2': 5.0, 'NOx': 5.0}
  ('PL-03', 'S-01'): {'PM2.5': 5.0, 'SO2': 5.0, 'NOx': 5.0}
  ('PL-03', 'S-02'): {'PM2.5': 5.0, 'SO2': 5.0, 'NOx': 5.0}
  ('PL-03', 'S-03'): {'PM2.5': 5.0, 'SO2': 5.0, 'NOx': 5.0}
  ('LOC-DEL-01', 'A-01'): {'PM2.5': 1.0, 'SO2': 2.0, 'NOx': 2.0}
  ('LOC-DEL-02', 'A-02'): {'PM2.5': 1.0, 'SO2': 2.0, 'NOx': 2.0}
  ('LOC-MUM-01', 'A-01'): {'PM2.5': 1.0, 'SO2': 2.0, 'NOx': 2.0}
  ('LOC-BLR-01', 'A-01'): {'PM2.5': 1.0, 'SO2': 2.0, 'NOx': 2.0}

PM2.5: 1352 BDL values replaced with LOD/2
SO2: 1350 BDL values replaced with LOD/2
NOx: 1359 BDL values replaced with LOD/2


### Rule 14 — Convert mg/Nm3 to ug/m3
**Why?** Some sensors report in mg/Nm3 instead of ug/m3.  
`1 mg = 1000 ug`, so multiply pollutant values by 1000 and fix the unit.

In [317]:
# ── SEE ──
mg_mask = df['Unit'] == 'mg/Nm3'
print(f'Rows with mg/Nm3: {mg_mask.sum()}')
if mg_mask.any():
    print(f'\nExample BEFORE conversion:')
    print(df.loc[mg_mask, ['Record_ID', 'PM2.5', 'SO2', 'NOx', 'Unit']].head(3).to_string())

Rows with mg/Nm3: 2664

Example BEFORE conversion:
   Record_ID  PM2.5    SO2    NOx    Unit
2     E00003  123.9    NaN  101.3  mg/Nm3
4     E00005   37.0   15.0  157.1  mg/Nm3
19    E00020   71.9  102.4  133.6  mg/Nm3


In [318]:
# ── CLEAN ──
for col in ['PM2.5', 'SO2', 'NOx']:
    old_vals = df.loc[mg_mask, col].copy()
    numeric = pd.to_numeric(df.loc[mg_mask, col], errors='coerce')
    df.loc[mg_mask, col] = (numeric * 1000).round(1).astype(str)
    
    for idx in df[mg_mask].index:
        audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': col,
                          'Old': str(old_vals[idx]), 'New': df.at[idx, col], 'Rule': 'R14'})

# fix the unit column too
for idx in df[mg_mask].index:
    audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': 'Unit',
                      'Old': 'mg/Nm3', 'New': 'ug/m3', 'Rule': 'R14'})
df.loc[mg_mask, 'Unit'] = 'ug/m3'

print(f'Converted {mg_mask.sum()} rows: mg/Nm3 -> ug/m3 (values x1000)')
print(f'\nExample AFTER:')
print(df.loc[mg_mask.index[mg_mask][:3], ['Record_ID', 'PM2.5', 'SO2', 'NOx', 'Unit']].to_string())

Converted 2664 rows: mg/Nm3 -> ug/m3 (values x1000)

Example AFTER:
   Record_ID     PM2.5       SO2       NOx   Unit
2     E00003  123900.0       nan  101300.0  ug/m3
4     E00005   37000.0   15000.0  157100.0  ug/m3
19    E00020   71900.0  102400.0  133600.0  ug/m3


In [319]:
# ── Now convert all pollutant columns to numeric (float) ──
for col in ['PM2.5', 'SO2', 'NOx']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('All pollutant columns are now float64')
print(f'\nUnit values: {df["Unit"].value_counts().to_dict()}')

All pollutant columns are now float64

Unit values: {'ug/m3': 28800}


---
## PHASE E: Calibration
Adjust sensor readings using drift and span factors.

### Rule 7 — Apply Zero Drift & Span Correction
**Why?** Sensors drift over time. `sensor_master` has the correction factors.  
**Formula:** `corrected = (raw - zero_drift) / span_mult`

In [320]:
# ── SEE: Calibration factors from sensor_master ──
print('Calibration factors per sensor:')
print(sensors[['Plant_ID', 'Stack_ID', 'Zero_Drift', 'Span_Mult']].to_string())
print()

Calibration factors per sensor:
     Plant_ID Stack_ID  Zero_Drift  Span_Mult
0       PL-01     S-01         0.5       1.02
1       PL-01     S-02         0.2       0.98
2       PL-02     S-01        -0.5       1.05
3       PL-03     S-01         0.8       1.10
4       PL-03     S-02         0.1       1.00
5       PL-03     S-03         0.0       1.01
6  LOC-DEL-01     A-01         0.1       1.00
7  LOC-DEL-02     A-02         0.0       0.99
8  LOC-MUM-01     A-01        -0.1       1.02
9  LOC-BLR-01     A-01         0.2       1.00



In [321]:
# ── CLEAN ──
# build lookup
cal_map = {}
for _, row in sensors.iterrows():
    cal_map[(row['Plant_ID'], row['Stack_ID'])] = {
        'drift': row['Zero_Drift'], 'span': row['Span_Mult']
    }

cal_count = 0
for col in ['PM2.5', 'SO2', 'NOx']:
    for idx in df.index:
        key = (df.at[idx, 'Plant_ID'], df.at[idx, 'Stack_ID'])
        cal = cal_map.get(key)
        if cal is None or pd.isna(df.at[idx, col]):
            continue
        
        drift = cal['drift']
        span = cal['span']
        
        if drift != 0 or span != 1.0:
            old_val = df.at[idx, col]
            new_val = round((old_val - drift) / span, 2)
            df.at[idx, col] = new_val
            audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': col,
                              'Old': old_val, 'New': new_val, 'Rule': 'R7'})
            cal_count += 1

print(f'Calibration adjustments made: {cal_count}')

Calibration adjustments made: 73900


---
## PHASE F: Outlier Detection

### Rule 8 — Cap Physical Spikes
**Why?** Values > 5000 ug/m3 are physically impossible from CEMS sensors.  
We cap these at the 99th percentile of valid data.

In [322]:
# ── SEE & CLEAN ──
for col in ['PM2.5', 'SO2', 'NOx']:
    spike_mask = df[col] > 5000
    count = spike_mask.sum()
    
    if count > 0:
        # calculate cap value from valid data
        valid = df.loc[(df[col].notna()) & (df[col] <= 5000), col]
        cap = round(valid.quantile(0.99), 1)
        
        print(f'{col}: {count} spikes found (max={df.loc[spike_mask, col].max():.0f})')
        print(f'  -> Capping at 99th percentile: {cap}')
        
        for idx in df[spike_mask].index:
            audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': col,
                              'Old': df.at[idx, col], 'New': cap, 'Rule': 'R8'})
        df.loc[spike_mask, col] = cap

PM2.5: 3069 spikes found (max=10203061)
  -> Capping at 99th percentile: 273.8
SO2: 3124 spikes found (max=10203061)
  -> Capping at 99th percentile: 275.7
NOx: 3107 spikes found (max=10203061)
  -> Capping at 99th percentile: 267.0


---
## PHASE G: Geographic Validation

### Rule 4 — Fix Invalid Coordinates
**Why?** Lat/Lon can have semicolons instead of commas, or lat/lon swapped.  
India's bounds: Lat 6-37, Lon 68-97.5.

In [323]:
# ── SEE & CLEAN ──
LAT_MIN, LAT_MAX = 6.0, 37.0
LON_MIN, LON_MAX = 68.0, 97.5
geo_fixes = 0

for idx in df.index:
    ll = df.at[idx, 'Lat_Lon']
    if pd.isna(ll):
        continue
    
    ll_str = str(ll).strip()
    old_val = ll_str
    fixed = False
    
    # fix semicolon
    if ';' in ll_str:
        ll_str = ll_str.replace(';', ',')
        fixed = True
    
    try:
        parts = ll_str.split(',')
        lat, lon = float(parts[0]), float(parts[1])
        
        # check if out of India bounds
        if lat < LAT_MIN or lat > LAT_MAX or lon < LON_MIN or lon > LON_MAX:
            # try swapping
            if LON_MIN <= lat <= LON_MAX and LAT_MIN <= lon <= LAT_MAX:
                lat, lon = lon, lat
                ll_str = f'{lat},{lon}'
                fixed = True
            else:
                df.at[idx, 'Lat_Lon'] = np.nan
                audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': 'Lat_Lon',
                                  'Old': old_val, 'New': 'NaN', 'Rule': 'R4'})
                geo_fixes += 1
                continue
        
        if fixed:
            df.at[idx, 'Lat_Lon'] = f'{lat},{lon}'
            audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': 'Lat_Lon',
                              'Old': old_val, 'New': df.at[idx, 'Lat_Lon'], 'Rule': 'R4'})
            geo_fixes += 1
    except:
        df.at[idx, 'Lat_Lon'] = np.nan
        geo_fixes += 1

print(f'Coordinates fixed/nulled: {geo_fixes}')

Coordinates fixed/nulled: 514


---
## PHASE H: Status Mapping

### Rule 12 — Map Non-Standard Status to Canonical
**Why?** We want only: `OK`, `FAULT`, `MAINT`, `GAP_FILLED`, `UNKNOWN`.

In [324]:
# ── SEE ──
canonical = {'OK', 'FAULT', 'MAINT', 'GAP_FILLED', 'UNKNOWN'}
non_standard = df[~df['Status'].isin(canonical)]
print(f'BEFORE — Non-standard statuses: {len(non_standard)}')
print(non_standard['Status'].value_counts())
print()

# ── CLEAN ──
status_map = {
    'OFFLINE': 'FAULT', 'DOWN': 'FAULT',
    'ERROR 404': 'FAULT', 'ERROR': 'FAULT',
    'MAINTENANCE': 'MAINT', 'NAN': 'UNKNOWN'
}

mask = ~df['Status'].isin(canonical)
for idx in df[mask].index:
    old = df.at[idx, 'Status']
    new = status_map.get(old, 'FAULT')
    df.at[idx, 'Status'] = new
    audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': 'Status',
                      'Old': old, 'New': new, 'Rule': 'R12'})

# ── CHECK ──
print('AFTER — Status values:')
print(df['Status'].value_counts())

BEFORE — Non-standard statuses: 4114
Status
OFFLINE        1136
DOWN           1117
ERROR           615
ERROR 404       571
MAINTENANCE     555
NAN             120
Name: count, dtype: int64

AFTER — Status values:
Status
FAULT         10157
OK             8313
MAINT          7795
GAP_FILLED     2415
UNKNOWN         120
Name: count, dtype: int64


---
## PHASE I: Cross-Table Joins

### Rule 13 — Validate Plant+Stack Combos

In [325]:
# Check every (Plant_ID, Stack_ID) exists in sensor_master
valid_combos = set(zip(sensors['Plant_ID'], sensors['Stack_ID']))
df_combos = set(zip(df['Plant_ID'].dropna(), df['Stack_ID'].dropna()))
invalid = df_combos - valid_combos
print(f'Invalid Plant+Stack combos: {len(invalid)}')
if invalid:
    print(f'  {invalid}')

Invalid Plant+Stack combos: 0


### Rule 15 — Tag Source Type

In [326]:
# Add Source_Type from sensor_master (Stack vs Ambient)
source_map = dict(zip(
    zip(sensors['Plant_ID'], sensors['Stack_ID']),
    sensors['Source_Type']
))

df['Source_Type'] = df.apply(
    lambda r: source_map.get((r['Plant_ID'], r['Stack_ID']), 'Unknown'), axis=1
)

print('Source_Type added:')
print(df['Source_Type'].value_counts())

Source_Type added:
Source_Type
Stack      15801
Ambient    10584
Unknown     2415
Name: count, dtype: int64


### Rule 10 — Apply Maintenance Windows
**Why?** If a sensor was being serviced, its readings during that time are unreliable.  
We null the pollutant data and set `Status = MAINT`.

In [327]:
# Parse maintenance timestamps
maint['Maint_Start'] = pd.to_datetime(maint['Maint_Start'], format='%d-%m-%Y %H:%M')
maint['Maint_End'] = pd.to_datetime(maint['Maint_End'], format='%d-%m-%Y %H:%M')

print('Maintenance windows:')
print(maint[['Plant_ID', 'Stack_ID', 'Maint_Start', 'Maint_End']].to_string())
print()

maint_count = 0
for _, m in maint.iterrows():
    mask = (
        (df['Plant_ID'] == m['Plant_ID']) &
        (df['Stack_ID'] == m['Stack_ID']) &
        (df['TS'] >= m['Maint_Start']) &
        (df['TS'] <= m['Maint_End']) &
        (df['Status'] != 'MAINT')
    )
    count = mask.sum()
    if count > 0:
        for col in ['PM2.5', 'SO2', 'NOx']:
            for idx in df[mask].index:
                audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': col,
                                  'Old': df.at[idx, col], 'New': 'NaN', 'Rule': 'R10'})
            df.loc[mask, col] = np.nan
        
        for idx in df[mask].index:
            audit_log.append({'Record_ID': df.at[idx, 'Record_ID'], 'Column': 'Status',
                              'Old': df.at[idx, 'Status'], 'New': 'MAINT', 'Rule': 'R10'})
        df.loc[mask, 'Status'] = 'MAINT'
        maint_count += count

print(f'Rows affected by maintenance: {maint_count}')

Maintenance windows:
      Plant_ID Stack_ID         Maint_Start           Maint_End
0   LOC-DEL-02     A-02 2025-01-21 23:00:00 2025-01-22 00:00:00
1        PL-01     S-02 2025-01-02 00:00:00 2025-01-02 01:00:00
2        PL-01     S-01 2025-01-05 11:00:00 2025-01-05 13:00:00
3        PL-01     S-02 2025-01-23 13:00:00 2025-01-23 16:00:00
4        PL-03     S-01 2025-01-24 07:00:00 2025-01-24 09:00:00
5   LOC-MUM-01     A-01 2025-01-21 23:00:00 2025-01-22 03:00:00
6   LOC-DEL-02     A-02 2025-01-02 19:00:00 2025-01-02 23:00:00
7   LOC-DEL-02     A-02 2025-01-06 19:00:00 2025-01-06 20:00:00
8   LOC-BLR-01     A-01 2025-01-02 09:00:00 2025-01-02 10:00:00
9        PL-03     S-01 2025-01-23 13:00:00 2025-01-23 16:00:00
10       PL-03     S-03 2025-01-23 14:00:00 2025-01-23 17:00:00
11  LOC-BLR-01     A-01 2025-01-22 03:00:00 2025-01-22 06:00:00
12  LOC-MUM-01     A-01 2025-01-27 19:00:00 2025-01-27 21:00:00
13       PL-01     S-01 2025-01-12 10:00:00 2025-01-12 11:00:00
14  LOC-BLR-01     

---
## PHASE J: QC & Compliance

### Rule 16 — Double-Entry QC Check
**Why?** Lab samples are entered twice. If they differ by >1%, it's a data entry error.

In [328]:
manual['Diff_Pct'] = abs(
    manual['Lab_PM25_Entry1'] - manual['Lab_PM25_Entry2']
) / manual['Lab_PM25_Entry1'] * 100

manual['QC_Status'] = manual['Diff_Pct'].apply(
    lambda d: 'QC_FAIL' if d > 1.0 else 'QC_PASS'
)

print('Double-entry QC results:')
print(manual[['Log_ID', 'Lab_PM25_Entry1', 'Lab_PM25_Entry2', 'Diff_Pct', 'QC_Status']].to_string())
print(f'\nQC Failures: {(manual["QC_Status"] == "QC_FAIL").sum()}')

Double-entry QC results:
   Log_ID  Lab_PM25_Entry1  Lab_PM25_Entry2    Diff_Pct QC_Status
0   L0001             98.7            987.0  900.000000   QC_FAIL
1   L0002            156.4            156.7    0.191816   QC_PASS
2   L0003             49.6             49.2    0.806452   QC_PASS
3   L0004            123.7            124.2    0.404204   QC_PASS
4   L0005            132.9            132.8    0.075245   QC_PASS
5   L0006             69.9            699.0  900.000000   QC_FAIL
6   L0007             86.8             87.1    0.345622   QC_PASS
7   L0008            149.0            149.3    0.201342   QC_PASS
8   L0009            122.2            122.0    0.163666   QC_PASS
9   L0010             44.4             44.5    0.225225   QC_PASS
10  L0011             17.4            174.0  900.000000   QC_FAIL
11  L0012            149.1            148.9    0.134138   QC_PASS
12  L0013             90.2             90.1    0.110865   QC_PASS
13  L0014             49.4             49.0    0.80

### Rule 19 — Flag Threshold Exceedances

In [329]:
# Build threshold lookup
print('Regulatory thresholds:')
print(thresholds.to_string())
print()

limit_map = {}
for _, t in thresholds.iterrows():
    limit_map[(t['Pollutant'], t['Source_Type'])] = t['Legal_Limit_ugm3']

# Flag exceedances
df['Exceedance_Flag'] = 'OK'
exc_count = 0
for col in ['PM2.5', 'SO2', 'NOx']:
    for idx in df.index:
        val = df.at[idx, col]
        src = df.at[idx, 'Source_Type']
        if pd.isna(val) or src == 'Unknown':
            continue
        limit = limit_map.get((col, src))
        if limit and val > limit:
            df.at[idx, 'Exceedance_Flag'] = 'EXCEEDANCE'
            exc_count += 1

print(f'Exceedances flagged: {exc_count}')
print(df['Exceedance_Flag'].value_counts())

Regulatory thresholds:
  Pollutant Source_Type  Legal_Limit_ugm3
0     PM2.5     Ambient              60.0
1     PM2.5       Stack             150.0
2       SO2     Ambient              80.0
3       SO2       Stack             200.0
4       NOx     Ambient              80.0
5       NOx       Stack             400.0

Exceedances flagged: 31872
Exceedance_Flag
EXCEEDANCE    16983
OK            11817
Name: count, dtype: int64


---
## PHASE K: Privacy

### Rule 18 — Redact PII from Inspection Notes
**Why?** Phone numbers and emails in notes are personal data. Must be redacted.

In [330]:
pii_count = 0
for idx in manual.index:
    note = str(manual.at[idx, 'Inspection_Notes'])
    old = note
    note = re.sub(r'\b\d{10}\b', '[REDACTED_PHONE]', note)
    note = re.sub(r'\S+@\S+\.\S+', '[REDACTED_EMAIL]', note)
    if note != old:
        manual.at[idx, 'Inspection_Notes'] = note
        pii_count += 1
        print(f'  {manual.at[idx, "Log_ID"]}:')
        print(f'    BEFORE: {old}')
        print(f'    AFTER:  {note}')

print(f'\nNotes with PII redacted: {pii_count}')

  L0005:
    BEFORE: Inspection passed. Signed off by Vikram (vikram.s@gov.in, 9090909090).
    AFTER:  Inspection passed. Signed off by Vikram [REDACTED_EMAIL] [REDACTED_PHONE]).
  L0006:
    BEFORE: System broken. Call Amit at 9876543210 or amit@email.com.
    AFTER:  System broken. Call Amit at [REDACTED_PHONE] or [REDACTED_EMAIL]
  L0010:
    BEFORE: System broken. Call Amit at 9876543210 or amit@email.com.
    AFTER:  System broken. Call Amit at [REDACTED_PHONE] or [REDACTED_EMAIL]
  L0013:
    BEFORE: Sensor drift detected. Contact Priya at 8765432109 for calibration.
    AFTER:  Sensor drift detected. Contact Priya at [REDACTED_PHONE] for calibration.
  L0014:
    BEFORE: Laser replaced by tech Suresh. Reach him at suresh.p@company.org or 7654321098.
    AFTER:  Laser replaced by tech Suresh. Reach him at [REDACTED_EMAIL] or [REDACTED_PHONE].
  L0021:
    BEFORE: Laser replaced by tech Suresh. Reach him at suresh.p@company.org or 7654321098.
    AFTER:  Laser replaced by tech Su

---
## PHASE L: Audit Trail

### Rule 20 — SHA-256 Integrity Hash
**Why?** Each row gets a unique hash. If anyone tampers with the data later, the hash won't match.

In [331]:
def make_hash(row):
    content = '|'.join(str(v) for v in row.values)
    return hashlib.sha256(content.encode('utf-8')).hexdigest()

df['Audit_Hash'] = df.apply(make_hash, axis=1)

print(f'Hashes generated for {len(df)} rows')
print(f'Example: {df["Audit_Hash"].iloc[0][:40]}...')

Hashes generated for 28800 rows
Example: d71814caf8869de73c7cb0f4961d17dc5d876a10...


---
## Save Everything

In [332]:
OUT = '../cleaned/dev'
df.to_csv(f'{OUT}/raw_cems_data_cleaned.csv', index=False)
manual.to_csv(f'{OUT}/manual_entries_cleaned.csv', index=False)

# Save audit log
audit_df = pd.DataFrame(audit_log)
audit_df.to_csv(f'{OUT}/cleaning_log.csv', index=False)

print(f'Cleaned data saved: {len(df)} rows')
print(f'Audit log saved: {len(audit_df)} entries')
print(f'\nSample audit entries:')
print(audit_df.head(10).to_string())

Cleaned data saved: 28800 rows
Audit log saved: 124145 entries

Sample audit entries:
  Record_ID Column    Old    New Rule
0    E00002   Unit  µg/m³  ug/m3   R2
1    E00024   Unit  µg/m³  ug/m3   R2
2    E00025   Unit  µg/m³  ug/m3   R2
3    E00045   Unit  µg/m³  ug/m3   R2
4    E00048   Unit  µg/m³  ug/m3   R2
5    E00084   Unit  µg/m³  ug/m3   R2
6    E00088   Unit  µg/m³  ug/m3   R2
7    E00100   Unit  µg/m³  ug/m3   R2
8    E00110   Unit  µg/m³  ug/m3   R2
9    E00113   Unit  µg/m³  ug/m3   R2


---
## Final Summary

In [333]:
print('=' * 55)
print('  DATA CLEANING COMPLETE')
print('=' * 55)
print(f'  Original rows:     {original_row_count}')
print(f'  Cleaned rows:      {len(df)}')
print(f'  Audit log entries: {len(audit_log)}')
print(f'  Unit:              {df["Unit"].unique()}')
print(f'  Status values:     {df["Status"].value_counts().to_dict()}')
print(f'  Exceedance flags:  {df["Exceedance_Flag"].value_counts().to_dict()}')
print(f'  Source types:      {df["Source_Type"].value_counts().to_dict()}')
print('=' * 55)

  DATA CLEANING COMPLETE
  Original rows:     27121
  Cleaned rows:      28800
  Audit log entries: 124145
  Unit:              ['ug/m3']
  Status values:     {'FAULT': 10049, 'OK': 8222, 'MAINT': 7994, 'GAP_FILLED': 2415, 'UNKNOWN': 120}
  Exceedance flags:  {'EXCEEDANCE': 16983, 'OK': 11817}
  Source types:      {'Stack': 15801, 'Ambient': 10584, 'Unknown': 2415}
